In [35]:
import weakref
from abc import ABC, abstractmethod

ENV_LIST = []

class Game(ABC):
  def __init__(self):
    ref = weakref.ref(self)
    ENV_LIST.append(ref)

  @abstractmethod
  def translate(self, move):
    pass

  @abstractmethod
  def current_turn(self):
    pass

  @abstractmethod
  def get_state(self):
    pass

  @abstractmethod
  def is_valid(self, move):
    pass

  @abstractmethod
  def valid_moves(self):
    pass

  @abstractmethod
  def is_terminal(self):
    pass

  @abstractmethod
  def get_winner(self):
    pass

  @abstractmethod
  def make_move(self, move):
    pass

  @abstractmethod
  def get_canonical_state(self):
    pass

  @abstractmethod
  def __repr__(self):
    pass

  @abstractmethod
  def __str__(self):
    pass

  @abstractmethod
  def copy(self):
    pass


In [36]:
from enum import IntEnum

class TTTGame(Game):
  class Position(IntEnum):
    TOPLEFT = 0
    TOPCENTER = 1
    TOPRIGHT = 2
    MIDDLELEFT = 3
    MIDDLECENTER = 4
    MIDDLERIGHT = 5
    BOTTOMLEFT = 6
    BOTTOMCENTER = 7
    BOTTOMRIGHT = 8

  def translate(self, move: Position) -> int:
    move = TTTGame.Position(move)
    return move.value

  def __init__(self, state=None, player_one_to_move=True, previous_move=None):
    super().__init__()
    if state is None:
      state = [0] * 9

    self.state = state
    self.turn = 1 if player_one_to_move else -1
    self.previous_move = previous_move

  def is_valid(self, move: Position) -> bool:
    try:
      TTTGame.Position(move)
    except ValueError:
      return False
    return self.state[move] == 0

  def valid_moves(self) -> list[Position]:
    return [i for i in range(9) if self.is_valid(i)]

  def is_terminal(self) -> bool:
    return self.get_winner() != 0 or len(self.valid_moves()) == 0

  def current_turn(self):
    return self.turn

  def get_state(self):
    return self.state

  def get_winner(self):
    occupant = self.state[self.Position.TOPLEFT]
    if occupant != 0:
      if self.state[self.Position.TOPCENTER] == occupant and self.state[self.Position.TOPRIGHT] == occupant:
        return occupant
      if self.state[self.Position.MIDDLELEFT] == occupant and self.state[self.Position.BOTTOMLEFT] == occupant:
        return occupant

    occupant = self.state[self.Position.BOTTOMRIGHT]
    if occupant != 0:
      if self.state[self.Position.BOTTOMCENTER] == occupant and self.state[self.Position.BOTTOMLEFT] == occupant:
        return occupant
      if self.state[self.Position.MIDDLERIGHT] == occupant and self.state[self.Position.TOPRIGHT] == occupant:
        return occupant

    occupant = self.state[self.Position.MIDDLECENTER]
    if occupant != 0:
      if self.state[self.Position.MIDDLELEFT] == occupant and self.state[self.Position.MIDDLERIGHT] == occupant:
        return occupant
      if self.state[self.Position.TOPLEFT] == occupant and self.state[self.Position.BOTTOMRIGHT] == occupant:
        return occupant
      if self.state[self.Position.TOPCENTER] == occupant and self.state[self.Position.BOTTOMCENTER] == occupant:
        return occupant
      if self.state[self.Position.TOPRIGHT] == occupant and self.state[self.Position.BOTTOMLEFT] == occupant:
        return occupant

    return 0

  def make_move(self, move: Position):
    self.state[move] = self.turn
    self.turn = -self.turn
    self.previous_move = move
    return self

  # Canonical form of state is always from the perspective of 1 with -1 being the opponent
  def get_canonical_state(self) -> list[Position]:
    norm_state = [s * self.turn for s in self.state]
    pos_mask = [0] * len(norm_state)
    for mov in self.valid_moves():
      ind = self.translate(mov)
      pos_mask[ind] = 1
    return norm_state, pos_mask


  def state_to_string(state):
    if state == 0:
      return "□"
    elif state == 1:
      return "X"
    elif state == -1:
      return "O"
    else:
      raise ValueError("Invalid state")

  def __repr__(self):
    return str(self)

  def __str__(self):
    state_strings = [TTTGame.state_to_string(s) for s in self.state]
    out = ""
    i = 0

    for _ in range(3):
      for _ in range(3):
        out += state_strings[i]
        i += 1
      out += "\n"
    return out

  def copy(self):
    return TTTGame(self.state.copy(), (self.turn == 1), self.previous_move)


In [37]:
from typing import Tuple
from functools import cache

class UTTTGame(Game):
  class Position(IntEnum):
    TOPLEFT = 0
    TOPCENTER = 1
    TOPRIGHT = 2
    MIDDLELEFT = 3
    MIDDLECENTER = 4
    MIDDLERIGHT = 5
    BOTTOMLEFT = 6
    BOTTOMCENTER = 7
    BOTTOMRIGHT = 8

  def translate(self, move: Tuple[Position, Position]) -> int:
    board_id, action = UTTTGame.Position(move[0]), UTTTGame.Position(move[1])
    return board_id.value * 9 + action.value

  def get_winner_small_board(state):
    occupant = state[UTTTGame.Position.TOPLEFT]
    if occupant != 0 and occupant != 2:
      if state[UTTTGame.Position.TOPCENTER] == occupant and state[UTTTGame.Position.TOPRIGHT] == occupant:
        return occupant
      if state[UTTTGame.Position.MIDDLELEFT] == occupant and state[UTTTGame.Position.BOTTOMLEFT] == occupant:
        return occupant

    occupant = state[UTTTGame.Position.BOTTOMRIGHT]
    if occupant != 0 and occupant != 2:
      if state[UTTTGame.Position.BOTTOMCENTER] == occupant and state[UTTTGame.Position.BOTTOMLEFT] == occupant:
        return occupant
      if state[UTTTGame.Position.MIDDLERIGHT] == occupant and state[UTTTGame.Position.TOPRIGHT] == occupant:
        return occupant

    occupant = state[UTTTGame.Position.MIDDLECENTER]
    if occupant != 0 and occupant !=2:
      if state[UTTTGame.Position.MIDDLELEFT] == occupant and state[UTTTGame.Position.MIDDLERIGHT] == occupant:
        return occupant
      if state[UTTTGame.Position.TOPLEFT] == occupant and state[UTTTGame.Position.BOTTOMRIGHT] == occupant:
        return occupant
      if state[UTTTGame.Position.TOPCENTER] == occupant and state[UTTTGame.Position.BOTTOMCENTER] == occupant:
        return occupant
      if state[UTTTGame.Position.TOPRIGHT] == occupant and state[UTTTGame.Position.BOTTOMLEFT] == occupant:
        return occupant

    empty_slots = [pos for pos in UTTTGame.Position if state[pos] == 0]
    if len(empty_slots) == 0:
      return 2

    return 0

  def __init__(self, state=None, player=1, previous_move=None, meta_state=None):
    if player != 1 and player != -1:
      raise ValueError("Invalid player")

    self.turn = player
    self.previous_move = previous_move

    if state is None:
      self.state = [0] * 81
      self.meta_state = [0] * 9
    else:
      if len(state) != 81: raise ValueError("Invalid state")
      self.state = state
      self.meta_state = self.calculate_meta_state() if meta_state == None else meta_state

    ref = weakref.ref(self)
    ENV_LIST.append(ref)

  def calculate_meta_state(self):
    meta_state = [0] * 9
    for i in range(9):
      meta_state[i] = UTTTGame.get_winner_small_board(self.state[i*9:i*9+9])
    return meta_state

  def update_meta_state(self, move):
    board_id = move[0]
    if self.meta_state[board_id] != 0:
      raise ValueError("Board already won")

    self.meta_state[board_id] = UTTTGame.get_winner_small_board(self.state[board_id*9:board_id*9+9])

  def current_turn(self):
    return self.turn

  def get_state(self):
    return self.state

  def is_valid(self, move: Tuple[Position, Position]) -> bool:
    try:
      idx = self.translate(move)
    except ValueError:
      return False
    if self.previous_move is None:
      return self.state[idx] == 0
    if move[0] == self.previous_move[1]:
      return self.state[idx] == 0
    return False

  def valid_moves(self) -> list[Tuple[Position, Position]]:
    finished_boards = [i for i, e in enumerate(self.meta_state) if e != 0 ]
    board_id = self.previous_move[1] if self.previous_move is not None else None

    if self.previous_move is None or self.meta_state[board_id] != 0:
      return [(i, j) for i in range(9) for j in range(9) if i not in finished_boards and self.state[self.translate((i, j))] == 0 ]
    moves = [(board_id, i) for i in range(9) if self.state[self.translate((board_id, i))] == 0]
    if len(moves) == 0:
      raise ValueError("No valid moves")
    return moves

  def is_terminal(self) -> bool:
    return self.get_winner() != 0 or len(self.valid_moves()) == 0

  def get_winner(self):
    res = UTTTGame.get_winner_small_board(self.meta_state)
    return 0 if res == 2 else res

  def make_move(self, move: Tuple[Position, Position]):
    self.state[self.translate(move)] = self.turn
    self.previous_move = move
    self.turn = -self.turn

    self.update_meta_state(move)
    return self

  def make_random_move(self):
    moves = self.valid_moves()
    move = random.choice(moves)
    self.make_move(move)

  def get_canonical_state(self) -> list[Position]:
    norm_state = [s * self.turn for s in self.state]
    pos_mask = [0] * len(norm_state)
    for mov in self.valid_moves():
      ind = self.translate(mov)
      pos_mask[ind] = 1
    return norm_state, pos_mask

  def state_to_string(state):
    if state == 0:
      return "□"
    elif state == 1:
      return "X"
    elif state == -1:
      return "O"
    else:
      return str(state)


  def __repr__(self):
    return "\n".join(("State: " + str(self.state),
                      "Previous Move: " + str(self.previous_move),
                      "Current Player: " + str(self.turn)))

  def __str__(self):
    state_copy = self.state.copy()
    index_order = [0, 1, 2, 9, 10, 11, 18, 19, 20, 3, 4, 5, 12, 13, 14, 21, 22, 23, 6, 7, 8, 15, 16, 17, 24, 25, 26,
                   27, 28, 29, 36, 37, 38, 45, 46, 47, 30, 31, 32, 39, 40, 41, 48, 49, 50, 33, 34, 35, 42, 43, 44, 51, 52, 53,
                   54, 55, 56, 63, 64, 65, 72, 73, 74, 57, 58, 59, 66, 67, 68, 75, 76, 77, 60, 61, 62, 69, 70, 71, 78, 79, 80]
    x_pattern = ["▪", " ", "▪", " ", "▪", " ", "▪", " ", "▪"]
    o_pattern = ["▪", "▪", "▪", "▪", " ", "▪", "▪", "▪", "▪"]
    t_pattern = [" ", " ", " ", " ", "▪", " ", " ", " ", " "]

    for i, board_result in enumerate(self.meta_state):
      if board_result == 1:
        state_copy[i*9:i*9+9] = x_pattern
      elif board_result == -1:
        state_copy[i*9:i*9+9] = o_pattern
      elif board_result == 2:
        state_copy[i*9:i*9+9] = t_pattern

    state_strings = [UTTTGame.state_to_string(state_copy[i]) for i in index_order]

    out = ""
    i = 0

    for w in range(3):
      for z in range(3):
        for y in range(3):
          for x in range(3):
            out += state_strings[i]
            i += 1
          out += " "
        out += "\n"
      out += "\n"
    return out

  def copy(self):
    return UTTTGame(self.state.copy(), self.turn, self.previous_move, self.meta_state.copy())

In [38]:
import random
import weakref
from math import sqrt, log, exp
from copy import deepcopy
from time import time

NODE_LIST = []

class Node:
  def __init__(self, environment, terminal, parent, action):
    self.total_reward = 0
    self.visit_count = 0

    self.child = None
    self.environment = environment
    self.terminal = terminal
    self.parent = weakref.ref(parent) if parent else None
    self.action = action

    ref = weakref.ref(self)
    NODE_LIST.append(ref)

  def _ucb(self):
    if self.visit_count == 0:
      return float('inf')

    parent_node = self.parent()
    return (self.total_reward / self.visit_count) + sqrt(1 * log(parent_node.visit_count) / self.visit_count)

  def _create_child(self):
    if self.terminal:
      return

    actions = self.environment.valid_moves()
    environments = [ self.environment.copy() for action in actions]
    for action, environment in zip(actions, environments):
      environment.make_move(action)

    # Get this class type
    NodeVariant = type(self)

    self.child = { action: NodeVariant(environment, environment.is_terminal(), self, action) for action, environment in zip(actions, environments)}

  def _rollout(self):
    new_env = self.environment.copy()
    while not new_env.is_terminal():
      actions = new_env.valid_moves()
      action = random.choice(actions)
      new_env.make_move(action)
    reward = new_env.get_winner() * self.environment.current_turn()
    return -reward

  def _traverse_to_leaf(node):
    while node.child:
      ucb_scores = {a: c._ucb() for a, c in node.child.items()}
      max_ucb = max(ucb_scores.values())
      actions = [a for a, s in ucb_scores.items() if s == max_ucb]

      action = random.choice(actions)
      node = node.child[action]
    return node

  def _update_parents(node, reward):
    flip = -1
    curr = node
    while curr.parent:
      curr = curr.parent()
      curr.visit_count += 1
      curr.total_reward += reward * flip
      flip *= -1

  def explore(self, perf=None):
    root = self

    start = time()
    current = Node._traverse_to_leaf(root)
    end = time()
    traverse_time = end - start

    start = time()
    reward = current._rollout()
    end = time()
    rollout_time = end - start

    current.total_reward += reward
    current.visit_count += 1

    start = time()
    Node._update_parents(current, reward)
    end = time()
    update_time = end - start

    start = time()
    current._create_child()
    end = time()
    create_time = end - start

    if perf is not None:
      perf["traverse_time"] = traverse_time
      perf["rollout_time"] = rollout_time
      perf["update_time"] = update_time
      perf["create_time"] = create_time

  def get_policy(self):
    if self.terminal:
      raise ValueError("Terminal node")

    if not self.child:
      raise ValueError("No children")

    policy = [0] * len(self.environment.get_state())
    for node in self.child.values():
      policy[self.environment.translate(node.action)] = node.visit_count

    # modified softmax
    sum_p = sum(policy)
    policy = [p / sum_p for p in policy]

    return policy

  def get_most_visited(self):
    if self.terminal:
      raise ValueError("Terminal node")

    if not self.child:
      raise ValueError("No children")

    visit_list = [node.visit_count for node in self.child.values()]
    max_visit = max(visit_list)

    most_visited_nodes = [c for c in self.child.values() if c.visit_count == max_visit]
    return random.choice(most_visited_nodes)

  def detach_parent(self):
    self.parent = None



In [151]:
class NeuralNode(Node):
  model = None
  build_tensor = None

  def __init__(self, environment, terminal, parent, action):
    super().__init__(environment, terminal, parent, action)
    self.neural_policy = None

  # Sets Node to use pytorch model
  # closure should take a node as input and return the tensor that should be
  # fed into the model for this node
  @classmethod
  def set_model(cls, model, closure):
    if not isinstance(model, torch.nn.Module):
      raise ValueError("Model is not a PyTorch module")

    cls.model = model
    cls.build_tensor = closure

  def _rollout(self):
    if self.environment.is_terminal():
      return -(self.environment.get_winner() * self.environment.current_turn())

    tensor = NeuralNode.build_tensor(self)
    with torch.no_grad():
      NeuralNode.model.eval()
      policy, value = NeuralNode.model(tensor)
      policy = policy.detach().numpy()[0]
      value = value.detach().numpy()[0][0]


    self.neural_policy = policy

    reward = value * self.environment.current_turn()
    return -reward

  def _ucb(self):
    if self.visit_count == 0:
      return float('inf')

    parent_node = self.parent()
    value_score = self.total_reward / self.visit_count
    action_index = self.environment.translate(self.action)
    exploration_score = parent_node.neural_policy[action_index] * sqrt(log(parent_node.visit_count) / self.visit_count)
    return value_score + exploration_score




In [40]:
def print_tree(root_node, indent=0):
    print('  ' * indent + f"- Environment: {root_node.environment.get_state()}, Visits: {root_node.visit_count}, Reward: {root_node.total_reward:.2f}")
    if root_node.child:
        for action, child_node in root_node.child.items():
            print('  ' * (indent + 1) + f"Action: {action}")
            print_tree(child_node, indent + 2)

In [41]:
from functools import reduce

def mcts(node, iters=100, show_execution_time=False, pause_after_iter=None):
  execution_times = []

  for i in range(iters):
    perf = {}
    node.explore(perf)
    execution_times.append(perf)
    if pause_after_iter != None and i >= pause_after_iter - 1:
      print_tree(node)
      input("Press Enter to continue")

  if show_execution_time:
    total_times = reduce(lambda x, y: {"traverse_time": x["traverse_time"] + y["traverse_time"],
                         "rollout_time": x["rollout_time"] + y["rollout_time"],
                         "update_time": x["update_time"] + y["update_time"],
                         "create_time": x["create_time"] + y["create_time"]}, execution_times, {"traverse_time": 0, "rollout_time": 0, "update_time": 0, "create_time": 0})
    print(f"Traverse time: {total_times['traverse_time'] / iters:.4f}")
    print(f"Rollout time: {total_times['rollout_time'] / iters:.4f}")
    print(f"Update time: {total_times['update_time'] / iters:.4f}")
    print(f"Create time: {total_times['create_time'] / iters:.4f}")

  policy = node.get_policy()
  next_node = node.get_most_visited()
  # next_node.detach_parent()

  return next_node, policy

In [31]:
env = TTTGame()

env.make_move(0)
env.make_move(6)
env.make_move(2)
env.make_move(8)
# env.make_move(3)

print(env)

env_copy = deepcopy(env)
root = Node(env_copy, env.is_terminal(), None, None)

n, _ = mcts(root, 100)
n.action

X□X
□□□
O□O



1

In [32]:
env = TTTGame()

env.make_move(0)
env.make_move(1)
env.make_move(3)

print(env)

env_copy = deepcopy(env)
root = Node(env_copy, env.is_terminal(), None, None)

n, _ = mcts(root, 1000)
n.action

XO□
X□□
□□□



6

In [33]:
env = TTTGame()

env.make_move(2)
env.make_move(5)
env.make_move(7)
env.make_move(1)

print(env)

env_copy = deepcopy(env)
root = Node(env_copy, env.is_terminal(), None, None)

n, _ = mcts(root, 1000)
n.action

□OX
□□O
□X□



6

In [42]:
class Player:
  def __init__(self, environment, is_first=True):
    self.environment = environment
    self.is_first = is_first

  def is_my_turn(self):
    return self.environment.current_turn() == (1 if self.is_first else -1)

  def play_turn(self):
    if not self.is_my_turn(): return None, None

    return self.on_my_turn()

  # Implement this func on subclasses
  # Should return a tuple (action, policy) for this node
  def on_my_turn(self):
    raise NotImplementedError

In [43]:
class Random_Player(Player):
  def __init__(self, environment, is_first=True):
    super().__init__(environment, is_first)

  def on_my_turn(self):
    action = random.choice(self.environment.valid_moves())
    policy = [0] * len(self.environment.get_state())
    for move in self.environment.valid_moves():
      policy[self.environment.translate(move)] = 1 / len(self.environment.valid_moves())
    return action, policy

In [44]:
class Human_Player(Player):
  def __init__(self, environment, is_first=True):
    super().__init__(environment, is_first)

  def on_my_turn(self):
    move = (-1, -1)
    print("Valid Moves:", self.environment.valid_moves())
    while not self.environment.is_valid(move):
      action = input("Enter a move: ")
      try:
        move = eval(action)
      except:
        pass
    return move, None

In [45]:
class MCTS_Player(Player):
  def __init__(self, environment, is_first=True, iter_count=100):
    super().__init__(environment, is_first)
    self.iter_count = iter_count

  def on_my_turn(self):
    env_copy = self.environment.copy()
    n = Node(env_copy, env_copy.is_terminal(), None, None)
    n, policy = mcts(n, self.iter_count)
    return n.action, policy


In [46]:
class Neural_MCTS_Player(Player):
  def __init__(self, environment, is_first=True, iter_count=100):
    super().__init__(environment, is_first)
    self.iter_count = iter_count

  def on_my_turn(self):
    env_copy = self.environment.copy()
    n = NeuralNode(env_copy, env_copy.is_terminal(), None, None)
    n, policy = mcts(n, self.iter_count)
    return n.action, policy

In [141]:
import torch
import torch.nn as nn

# Input Shape: [batch_size, 2, 81]
# 2 Channels: First channel show piece placements (Xs and Os), Second shows valid moves locations
class UTTTNet(nn.Module):
  def __init__(self):
    super(UTTTNet, self).__init__()
    self.conv1 = nn.Conv2d(in_channels=2, out_channels=128, kernel_size=3, stride=3, padding=0)
    self.bn1 = nn.BatchNorm2d(128)
    self.conv2 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=1, stride=1, padding=0)
    self.bn2 = nn.BatchNorm2d(256)
    self.conv3 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=0)
    self.bn3 = nn.BatchNorm2d(512)

    self.policy1 = nn.Linear(in_features=512, out_features=256)
    self.policy2 = nn.Linear(in_features=256, out_features=128)
    self.policy3 = nn.Linear(in_features=128, out_features=81)

    self.value1 = nn.Linear(in_features=512, out_features=256)
    self.value2 = nn.Linear(in_features=256, out_features=128)
    self.value3 = nn.Linear(in_features=128, out_features=1)

    self.softmax = nn.Softmax(dim=1)
    self.tanh = nn.Tanh()
    self.relu = nn.ReLU()

  def forward(self, x):
    batch_size = x.shape[0]
    leading_dim = x.shape[:-1]

    # mask = x[:, 1, :]

    x = x.view(*leading_dim, 3, 3, 3, 3)
    x = x.permute(*range(len(leading_dim)), -4, -2, -3, -1)
    x = x.contiguous()
    x = x.view(*leading_dim, 9, 9)

    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu(x)
    x = self.conv3(x)
    x = self.bn3(x)
    x = self.relu(x)

    x = x.view(batch_size, -1)

    p = self.policy1(x)
    p = self.relu(p)
    p = self.policy2(p)
    p = self.relu(p)
    p = self.policy3(p)
    p = self.softmax(p)
    # p = p * mask
    # p = p / p.sum(dim=1, keepdim=True)

    v = self.value1(x)
    v = self.relu(v)
    v = self.value2(v)
    v = self.relu(v)
    v = self.value3(v)
    v = self.tanh(v)

    return p, v

    return x

# NeuralNode closure for building input
def build_tensor(node):
  state, mask = node.environment.get_canonical_state()
  state = torch.Tensor(state)
  mask = torch.Tensor(mask)
  return torch.stack((state, mask), dim=0).unsqueeze(0)

In [135]:
class Arena:
  def __init__(self, environment, player_one, player_two):
    self.environment = environment

    self.player_one = player_one
    self.player_two = player_two

    self.record = {
      "player_one": {
          "states": [],
          "policies": []
      },
      "player_two": {
          "states": [],
          "policies": []
      },
      "winner": None
    }

  def play_game(self, show=False, record=False):
    if show: print(self.environment)
    while not self.environment.is_terminal():
      action1, policy1 = self.player_one.play_turn()
      action2, policy2 = self.player_two.play_turn()

      action = action1 if action1 != None else action2
      policy = policy1 if policy1 != None else policy2

      player = "player_one" if action1 != None else "player_two"

      if record:
        self.record[player]["states"].append(self.environment.get_canonical_state())
        self.record[player]["policies"].append(policy)

      self.environment.make_move(action)
      if show: print(self.environment)

    winner = self.environment.get_winner()
    if record:
      self.record["winner"] = winner
    if show: print(f"Winner: {winner}")
    return winner

  def get_record(self):
    return self.record



def create_arena(game_type='UTTT', player_one_type='human', player_two_type='mcts'):
  if game_type == 'UTTT':
    environment = UTTTGame()
  elif game_type == 'TTT':
    environment = TTTGame()
  else:
    raise ValueError("Game type does not exist")

  if player_one_type == 'random':
    player_one = Random_Player(environment, True)
  elif player_one_type.startswith('mcts'):
    iter_count = int(player_one_type.split('_')[1]) if player_one_type != 'mcts' else 100
    player_one = MCTS_Player(environment, True, iter_count)
  elif player_one_type == 'human':
    player_one = Human_Player(environment, True)
  elif player_one_type.startswith('nmcts'):
    iter_count = int(player_one_type.split('_')[1]) if player_one_type != 'nmcts' else 100
    player_one = Neural_MCTS_Player(environment, True, iter_count)
  else:
    raise ValueError("Player one type does not exist")

  if player_two_type == 'random':
    player_two = Random_Player(environment, False)
  elif player_two_type.startswith('mcts'):
    iter_count = int(player_two_type.split('_')[1]) if player_two_type != 'mcts' else 100
    player_two = MCTS_Player(environment, False, iter_count)
  elif player_two_type == 'human':
    player_two = Human_Player(environment, False)
  elif player_two_type.startswith('nmcts'):
    iter_count = int(player_two_type.split('_')[1]) if player_two_type != 'nmcts' else 100
    player_two = Neural_MCTS_Player(environment, False, iter_count)
  else:
    raise ValueError("Player two type does not exist")

  return Arena(environment, player_one, player_two)



In [154]:

records = []
winners = []

NeuralNode.set_model(model, build_tensor)

for i in range(20):
  print(f"Game {i+1}")
  arena = create_arena('UTTT', 'mcts_1000', 'mcts_1000')
  winner = arena.play_game(show=False, record=True)
  record = arena.get_record()
  records.append(record)
  winners.append(winner)


winrate1 = reduce(lambda x, y: x + int(y ==  1), winners, 0) / len(winners)
tierate  = reduce(lambda x, y: x + int(y ==  0), winners, 0) / len(winners)
winrate2 = reduce(lambda x, y: x + int(y == -1), winners, 0) / len(winners)

print(f"Player One winrate: {winrate1}")
print(f"Average tierate: {tierate}")
print(f"Player Two Winrate: {winrate2}")

print(records)

Game 1
Game 2
Game 3
Game 4
Game 5
Game 6
Game 7
Game 8
Game 9
Game 10
Game 11
Game 12
Game 13
Game 14
Game 15
Game 16
Game 17
Game 18
Game 19
Game 20
Player One winrate: 0.25
Average tierate: 0.4
Player Two Winrate: 0.35
[{'player_one': {'states': [([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), ([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0

In [156]:
# Save records object to csv
import csv

with open('records.csv', 'w', newline='') as csvfile:
  writer = csv.writer(csvfile)
  writer.writerow(["player_one_states", "player_one_policies", "player_two_states", "player_two_policies", "winner"])
  for record in records:
    writer.writerow([record["player_one"]["states"], record["player_one"]["policies"], record["player_two"]["states"], record["player_two"]["policies"], record["winner"]])

In [117]:
# start download
from google.colab import files
files.download('records.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [137]:
# Load records object from csv
import csv

records = []

with open('records.csv', 'r') as csvfile:
  reader = csv.reader(csvfile)
  next(reader)
  for row in reader:
    record = {
      "player_one": {
          "states": eval(row[0]),
          "policies": eval(row[1])
      },
      "player_two": {
          "states": eval(row[2]),
          "policies": eval(row[3])
      },
      "winner": eval(row[4])
    }
    records.append(record)

In [21]:
NOTNULLNODE = []

for ref in NODE_LIST:
  node = ref()
  if node is not None:
    NOTNULLNODE.append(ref)

print(len(NODE_LIST))
print(len(NOTNULLNODE))

189759
0


In [22]:
import sys
NOTNULLENV = []
NUM_REFS = []

for ref in ENV_LIST:
  env = ref()
  if env is not None:
    NOTNULLENV.append(ref)
    NUM_REFS.append(sys.getrefcount(env) - 1)

print(len(ENV_LIST))
print(len(NOTNULLENV))

if len(NUM_REFS) > 0:
  average_num_refs = sum(NUM_REFS) / len(NUM_REFS)
  print(f"Average number of references: {average_num_refs}")

  std_num_refs = (sum([(x - average_num_refs) ** 2 for x in NUM_REFS]) / len(NUM_REFS)) ** 0.5
  print(f"Standard deviation of number of references: {std_num_refs}")

  median_num_refs = sorted(NUM_REFS)[len(NUM_REFS) // 2]
  print(f"Median number of references: {median_num_refs}")

  min_num_refs = min(NUM_REFS)
  print(f"Minimum number of references: {min_num_refs}")

  max_num_refs = max(NUM_REFS)
  print(f"Maximum number of references: {max_num_refs}")





212760
0


In [138]:
from torch.utils.data import Dataset

class UTTTRecordDataset(Dataset):
  def __init__(self, records):
    # Each elem is state, mask, policy, reward
    self.data = [] # State, mask, and policy
    self.rewards = [] # Reward

    self.augmented_data = None

    for record in records:
      for state, policy in zip(record["player_one"]["states"], record["player_one"]["policies"]):
        self.data.append((state[0], state[1], policy))
        self.rewards.append(record['winner'])
      for state, policy in zip(record["player_two"]["states"], record["player_two"]["policies"]):
        self.data.append((state[0], state[1], policy))
        self.rewards.append(-record["winner"])

    self.data = torch.Tensor(self.data)
    self.rewards = torch.Tensor(self.rewards)

  def augment_data(self):
    # [dataset_size, 3, 81]
    def to_2d(x):
      dataset_size = x.shape[0]
      # Reshape flat tensor to 2D tensor
      x = x.view(dataset_size, 3, 3, 3, 3, 3)
      x = x.permute(0, 1, 2, 4, 3, 5)
      x = x.contiguous()
      x = x.view(dataset_size, 3, 9, 9)
      return x, dataset_size

    def to_flat(x, dataset_size):
      # Reshape back to flat
      x = x.contiguous()
      x = x.view(dataset_size, 3, 3, 3, 3, 3)
      x = x.permute(0, 1, 2, 4, 3, 5)
      x = x.contiguous()
      x = x.view(dataset_size, 3, 81)

      return x

    self.augmented_data = []
    curr, dataset_size = to_2d(self.data)
    for _ in range(4):
      self.augmented_data.append(curr)
      curr = torch.rot90(curr, 1, [0, 1])
      flip = torch.flip(curr, [1])
      self.augmented_data.append(flip)

    self.augmented_data = [ to_flat(x, dataset_size) for x in self.augmented_data ]
    self.augmented_data = torch.cat(self.augmented_data, dim=0)

  def __len__(self):
    return len(self.rewards) if self.augmented_data is None else len(self.augmented_data)

  # State, Policy, Reward
  def __getitem__(self, idx):
    if self.augmented_data is not None:
      return self.augmented_data[idx][0], self.augmented_data[idx][1], self.augmented_data[idx][2], self.rewards[idx % 8]
    return self.data[idx][0], self.data[idx][1], self.data[idx][2], self.rewards[idx]

In [97]:
test = []
a = torch.arange(81)
a = a.view(9, 9)
for _ in range(4):
  test.append(a)
  a = torch.rot90(a, 1, [0, 1])
  flip = torch.flip(a, [1])
  test.append(flip)

def check_any_equal(test):
  for i, t in enumerate(test):
    for j, t2 in enumerate(test[i+1:], i+1):
      if torch.equal(t, t2):
        print(f"Equal at index {i} and {j}")
        print(t)
        return True
  return False

print(check_any_equal(test))

False


In [155]:
from torch.utils.data import DataLoader, random_split

# Create the dataset
dataset = UTTTRecordDataset(records)
dataset.augment_data()

# Split the dataset into training and validation sets
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
print('Train Size:', len(train_dataset))
print('Val Size:', len(val_dataset))

# Create the dataloader
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Define the loss functions
criterion_policy = nn.CrossEntropyLoss() # Use CrossEntropyLoss for policy
criterion_value = nn.MSELoss()

model = UTTTNet()

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Train Size: 6585
Val Size: 1647


In [ ]:
# Training loop
num_epochs = 250
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for states, masks, policies, rewards in train_dataloader:
        # Forward pass
        predicted_policies, predicted_values = model(torch.stack((states, masks), dim=1))

        # Calculate the loss
        policy_loss = criterion_policy(predicted_policies, torch.argmax(policies, dim=1))
        value_loss = criterion_value(predicted_values.squeeze(), rewards)
        loss = policy_loss + value_loss

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * states.size(0)

    avg_train_loss = total_loss / len(train_dataset)
    if (epoch + 1) % 10 == 0:
      print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}")

    # Validation loop
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for states, masks, policies, rewards in val_dataloader:
            predicted_policies, predicted_values = model(torch.stack((states, masks), dim=1))
            policy_loss = criterion_policy(predicted_policies, torch.argmax(policies, dim=1))
            value_loss = criterion_value(predicted_values.squeeze(), rewards)
            loss = policy_loss + value_loss
            total_val_loss += loss.item() * states.size(0)

    avg_val_loss = total_val_loss / len(val_dataset)
    if (epoch + 1) % 10 == 0:
      print(f"Epoch [{epoch+1}/{num_epochs}], Val Loss: {avg_val_loss:.4f}")

print("Training finished.")

Epoch [10/250], Train Loss: 4.2056
Epoch [10/250], Val Loss: 4.2280
Epoch [20/250], Train Loss: 4.1778
Epoch [20/250], Val Loss: 4.2099
Epoch [30/250], Train Loss: 4.1473
Epoch [30/250], Val Loss: 4.1918
Epoch [40/250], Train Loss: 4.1392
Epoch [40/250], Val Loss: 4.1909
Epoch [50/250], Train Loss: 4.1348
Epoch [50/250], Val Loss: 4.1838
Epoch [60/250], Train Loss: 4.1282
Epoch [60/250], Val Loss: 4.1818
Epoch [70/250], Train Loss: 4.1221
Epoch [70/250], Val Loss: 4.1749
Epoch [80/250], Train Loss: 4.1188
Epoch [80/250], Val Loss: 4.1722
Epoch [90/250], Train Loss: 4.1201
Epoch [90/250], Val Loss: 4.1673
Epoch [100/250], Train Loss: 4.1007
Epoch [100/250], Val Loss: 4.1545
Epoch [110/250], Train Loss: 4.1031
Epoch [110/250], Val Loss: 4.1591
Epoch [120/250], Train Loss: 4.0970
Epoch [120/250], Val Loss: 4.1553
Epoch [130/250], Train Loss: 4.0925
Epoch [130/250], Val Loss: 4.1544
Epoch [140/250], Train Loss: 4.1031
Epoch [140/250], Val Loss: 4.1517
Epoch [150/250], Train Loss: 4.0899
Ep

In [90]:
batch_size = 6
board_states = []
for _ in range(batch_size):
  game = UTTTGame()
  for _ in range(20):
    game.make_random_move()
  board_states.append(game.get_canonical_state())

input = torch.Tensor(board_states)

# Create an instance of the model
model = UTTTNet()

# Pass the random tensor through the model
output = model(input)

# Print the shape of the output tensor
# print(f"Input tensor shape: {input.shape}")
# print(f"Output tensor shape: {output.shape}")

# You can also print the output values to see the results
print(f"Output tensor values: {output}")

Output tensor values: (tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.1448, 0.0000, 0.1374, 0.1408, 0.1537, 0.1425, 0.1366, 0.1441,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1377, 0.1159, 0.1171, 0.0000, 0.1297, 0.1283, 0.1299, 0.1221, 0.1192,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
    